In [2]:
import csv
import os
import bisect
import json
import sys
import numpy as np
from collections import Counter
from multiprocessing import Pool
from scipy.stats import linregress
import pandas as pd
import bisect
from collections import defaultdict
from calc_optimal import calc_optimal_allocation
import subprocess
import re

def get_aligned_size(size, alignment):
    return (size + alignment - 1) // alignment * alignment


def generate_alloc_sizes(factor, max_size, min_size, alignment=8):
    if max_size > 4 * 1024 * 1024:
        raise ValueError(f"maximum alloc size {max_size} is more than the slab size {1024 * 1024}")

    if factor <= 1.0:
        raise ValueError(f"invalid factor {factor}")

    alloc_sizes = set()
    size = min_size

    while size < max_size:
        n_per_slab = 4 * 1024 * 1024 // size  # Assuming Slab::kSize is 1MB
        if n_per_slab <= 1:
            break
        alloc_sizes.add(size)
        prev_size = size
        size = get_aligned_size(int(size * factor), alignment)
        if prev_size == size:
            raise ValueError(f"invalid incFactor {factor}")

    alloc_sizes.add(get_aligned_size(max_size, alignment))
    return alloc_sizes


class ReuseDistanceCalculator:
    def __init__(self):
        self.last_access_time = {}  # Maps elements to last access time
        self.access_times_tree = []  # Sorted list of access times
        self.histogram = defaultdict(int)
        self.current_time = 0
    
    def feed(self, element):
        reuse_distance = -1

        if element in self.last_access_time:
            last_time = self.last_access_time[element]
            idx = bisect.bisect_right(self.access_times_tree, last_time)
            reuse_distance = len(self.access_times_tree) - idx
            del self.access_times_tree[bisect.bisect_left(self.access_times_tree, last_time)]
        
        self.histogram[reuse_distance] += 1
        
        bisect.insort(self.access_times_tree, self.current_time)
        self.last_access_time[element] = self.current_time
        self.current_time += 1

        return reuse_distance

    def get_histogram(self):
        return dict(self.histogram)

    def reset_histogram(self):
        self.histogram.clear()
        

import numpy as np

class ReuseDistanceCalculator:
    def __init__(self):
        self.last_access_time = {}  # Maps elements to last access time
        self.access_times_tree = []  # Sorted list of access times
        self.histogram = defaultdict(int)
        self.current_time = 0

    def feed(self, element):
        """
        Feeds a new access element and updates reuse distance histogram.

        Args:
            element: The accessed element.
        
        Returns:
            int: The reuse distance for this access (-1 for first-time access).
        """
        reuse_distance = -1

        if element in self.last_access_time:
            last_time = self.last_access_time[element]
            idx = bisect.bisect_right(self.access_times_tree, last_time)
            reuse_distance = len(self.access_times_tree) - idx
            # Remove the old access time
            del self.access_times_tree[bisect.bisect_left(self.access_times_tree, last_time)]
        
        self.histogram[reuse_distance] += 1
        
        # Insert the current access time and update last seen time
        bisect.insort(self.access_times_tree, self.current_time)
        self.last_access_time[element] = self.current_time
        self.current_time += 1

        return reuse_distance

    def get_histogram(self):
        """Returns the current histogram of reuse distances."""
        return dict(self.histogram)

    def reset(self):
        """Resets the internal state for reuse with a new sequence."""
        self.__init__()

    def query_mrc(self, object_size, max_slab_cnt):
        reuse_distance_histogram = self.get_histogram()
        memory_sizes = np.arange(4 * 1024 * 1024, 4 * (max_slab_cnt + 1) * 1024 * 1024, 4 * 1024 * 1024)
        total_records = sum(reuse_distance_histogram.values())
        last_miss_ratio = 1
        mrc = {}
        mrc_delta = {}

        for memory_size in memory_sizes:
            slab_cnt = memory_size // (4 * 1024 * 1024)
            max_objects = memory_size // object_size

            miss_count = sum(
                count for reuse_distance, count in reuse_distance_histogram.items()
                if reuse_distance >= max_objects or reuse_distance == -1
            )

            miss_ratio = miss_count / total_records
            miss_ratio_delta = last_miss_ratio - miss_ratio
            mrc[slab_cnt] = miss_ratio
            mrc_delta[slab_cnt] = miss_ratio_delta
            last_miss_ratio = miss_ratio

        return mrc, mrc_delta


def read_csv_dict_line_by_line(file_path):
    with open(file_path, mode='r', newline='', encoding='utf-8') as csvfile:
        reader = csv.DictReader(csvfile)  # Automatically handles the header
        for row in reader:
            yield row  # Row is a dict with keys from the header


def process_trace(trace_file_path, alloc_sizes, max_slab, static, change_timings):
    
    reuse_distance_calculators = {alloc_size: ReuseDistanceCalculator() for alloc_size in alloc_sizes}
    nr_requests = {alloc_size: 0 for alloc_size in alloc_sizes}
    optimal_miss_ratios = []
    request_id = 0
    for row in read_csv_dict_line_by_line(trace_file_path):
        if not static and request_id in change_timings:
            mrc_dict, mrc_delta_dict = {}, {}
            for alloc_size in alloc_sizes:
                mrc, mrc_delta = reuse_distance_calculators[alloc_size].query_mrc(alloc_size, max_slab)
                mrc_dict[alloc_size] = mrc
                mrc_delta_dict[alloc_size] = mrc_delta
            
            r, _ = calc_optimal_allocation(
                mrc_dict, mrc_delta_dict, max_slab, alloc_sizes, [nr_requests[alloc_size] for alloc_size in alloc_sizes]
            )  
            optimal_miss_ratios.append((sum(nr_requests.values()), r))
            
            
            reuse_distance_calculators[alloc_size].reset_histogram()
            nr_requests[alloc_size] = 0
        
        object_id = row['object_id']
        object_size = int(row['object_size'])
        object_size = max(24, object_size)
        object_size += (32 + len(str(object_id)))
        index = bisect.bisect_left(alloc_sizes, object_size)
        if index < len(alloc_sizes):
            alloc_size = alloc_sizes[index]
            reuse_distance_calculators[alloc_size].feed(object_id)
            nr_requests[alloc_size] += 1
        
        request_id += 1
    
    return optimal_miss_ratios
            


In [ ]:
path = "/mydata/hongshu/traces/synth_dynamic_400.csv"
result = process_trace(path, [256, 512, 1024, 2048, 4096], 512, False, [40_000_000, 80_000_000])